# JEPA-CAD training on a Colab T4

The model is **~76.5M trainable / ~127M total** (per the 2026-07-25 architecture audit in
`AGENT_CROSSTALK.md`), *not* 24B despite the `space_24b` naming. A single T4 is sufficient.

Prior runs never exceeded 83 steps and all ran on CPU at 0.18-0.82 samples/sec. The point of
this notebook is to get past that: real GPU throughput, checkpointing to Drive so a runtime
kill costs minutes rather than the run.

**Set Runtime -> Change runtime type -> T4 GPU before running anything.**

In [ ]:
#@title 1. Confirm GPU
import torch, multiprocessing
assert torch.cuda.is_available(), 'No GPU -- Runtime > Change runtime type > T4 GPU'
p = torch.cuda.get_device_properties(0)
print(f'{p.name} | {p.total_memory/1e9:.1f} GB | cc {p.major}.{p.minor}')
print('torch', torch.__version__, '| vCPUs', multiprocessing.cpu_count())

In [ ]:
#@title 2. Code from GitHub (public repo, no token needed)
%cd /content
!rm -rf jepa-cad && git clone --depth 1 https://github.com/SaahithV6/jepa-cad.git
%cd /content/jepa-cad
!git log --oneline -3
# Colab ships torch; the training path additionally needs pyyaml (present) and numpy.
!pip install -q pyyaml

In [ ]:
#@title 3. Corpus from Drive
# Upload jepa-train-corpus.tar.gz (~1 GB) to Drive once. It contains three
# disciplines -- the third did not exist before and is what makes a
# "x kg payload to y km" specification conditionable at all:
#   artifacts/jepa-train-bundle/graph.json   (407 MB, 327,700 nodes)
#   artifacts/physics_shards/fea/*.npz       (9,029  CalculiX FRD stress fields)
#   artifacts/physics_shards/cfd/*.npz       (3,267  OpenFOAM velocity/pressure)
#   artifacts/physics_shards/traj/*.npz      (1,500  propulsion + trajectory)
#   artifacts/propulsion_trajectory/records.jsonl  (unnormalised design/outcome pairs)
#
# traj channels: thrust, mass, drag, mach, dynamic_pressure, accel, velocity, altitude
from google.colab import drive
drive.mount('/content/drive')

CORPUS = '/content/drive/MyDrive/cadflow/jepa-train-corpus.tar.gz'  #@param {type:"string"}
import os, pathlib
assert os.path.exists(CORPUS), f'not found: {CORPUS}'
!tar xzf "$CORPUS" -C /content/jepa-cad/

import glob
for kind in ('fea', 'cfd', 'traj'):
    n = len(glob.glob(f'/content/jepa-cad/artifacts/physics_shards/{kind}/*.npz'))
    print(f'{kind:5s} shards: {n}')
print('graph MB:', round(os.path.getsize(
    '/content/jepa-cad/artifacts/jepa-train-bundle/graph.json')/1e6))

In [ ]:
#@title 4. Checkpoint directory on Drive (survives runtime death)
import pathlib
CKPT = pathlib.Path('/content/drive/MyDrive/cadflow/checkpoints')
CKPT.mkdir(parents=True, exist_ok=True)
!ln -sfn "$CKPT" /content/jepa-cad/checkpoints
print('checkpoints ->', CKPT)
!ls -la "$CKPT" | head

In [ ]:
#@title 5. Short GPU smoke run -- prove throughput before committing hours
# physics_shards_only=true is justified: the config comments say to flip it once
# shard coverage passes ~5k, and the corpus now carries 12,293 shards.
#
# precision=fp16 is REQUIRED on a T4. The config asks for bf16, which needs
# Ampere or newer; the T4 is Turing (SM 7.5) and has no native bf16.
!python train.py --family space_24b --data-source graph --max-steps 20 \
    --set data.physics_shards_only=true \
    --set train.precision=fp16 2>&1 | tail -30

In [ ]:
#@title 6. Compare against the CPU baseline
# Historical runs (runs/*/metrics.jsonl) sat at 0.18-0.82 samples/sec on CPU.
# Anything below ~5 samples/sec here means the GPU is not actually being used --
# check that the model moved to cuda before assuming the data path is the bottleneck.
import json, pathlib, glob
for f in sorted(glob.glob('/content/jepa-cad/runs/*/metrics.jsonl')):
    lines = [json.loads(l) for l in open(f) if l.strip()]
    if not lines: continue
    last = lines[-1]
    print(f"{pathlib.Path(f).parent.name:28s} steps={last.get('step'):5d} "
          f"loss={last.get('loss'):.4f} samples/s={last.get('samples_per_sec'):.2f}")

In [ ]:
#@title 7. Real run
# Raise max-steps once cell 5 shows healthy throughput and a falling loss.
#
# Watch embed_std: latent JEPA can quietly collapse to a constant embedding, and
# collapse_std_threshold (0.01) is meant to catch it. That guard only works with
# batch_size > 1 -- at batch 1 the statistic is std over a single sample, i.e.
# NaN, and the check silently never fires. The config now uses batch 8 x accum 4
# (same effective batch of 32 as the original 1 x 32).
!python train.py --family space_24b --data-source graph --max-steps 2000 \
    --set data.physics_shards_only=true \
    --set train.precision=fp16 2>&1 | tail -40